# Preprocessing Pipeline for YOLO Training

## Preprocessing Steps Explained

- **Letterbox Resizing**  
  Resize each image to a fixed square size (e.g. 640×640) without distortion by scaling to fit and padding the shorter edges. Ensures all inputs share the same dimensions while preserving object shapes and centering content.

- **CLAHE (Contrast-Limited Adaptive Histogram Equalization)**  
  Enhances local contrast by applying histogram equalization on small image tiles, then clipping extreme amplification to avoid noise blow-up. Makes faint features (like flagellar motors) more visible in low-contrast tomogram slices.

- **NLMeans Denoising**  
  Reduces speckle and random noise by averaging each patch with similar patches found across the image, preserving textures and edges. Particularly effective for the grainy appearance of cryo-ET data.

- **Grayscale Normalization**  
  Stretches pixel intensities to the full 0–255 range, standardizing brightness and contrast across slices. A step before further enhancement and denoising.

Each of these operations boosts the signal-to-noise and ensures uniform, centered inputs for a YOLO detector, improving both training stability and detection accuracy.```


## References

[Data Preprocessing](https://docs.ultralytics.com/guides/preprocessing_annotated_data/)

[Data Augmentation](https://www.ultralytics.com/glossary/data-augmentation)



In [2]:
import os
import random
import shutil
import sys

import albumentations as A
import cv2
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from src import config

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
#  Imports & Configuration

# User parameters

if config.USE_SAMPLED_TRAIN_DATASET:
    INPUT_DIR = config.SAMPLED_TRAIN_DATASET_DIR  # path of sampled training dataset
else:
    INPUT_DIR = config.TRAIN_DATASET_DIR  # path of training dataset

PREPROCESSED_OUTPUT_DIR = config.PREPROCESSED_DATASET_DIR
TARGET_SIZE = (640, 640)  # (height, width)
EXTS = [".jpg", ".png", ".tif", ".tiff"]  # supported extensions

# CLAHE & denoise settings
CLAHE_CFG = {"clip_limit": 2.0, "grid_size": (8, 8)}
DENOISE_CFG = {"h": 10, "template_size": 7, "search_size": 21}

# Ensure output directory exists
os.makedirs(PREPROCESSED_OUTPUT_DIR, exist_ok=True)

In [3]:
# Helper Functions


def letterbox_resize(img, new_shape=TARGET_SIZE, color=(114, 114, 114)):
    """
    Resize+pad to new_shape, keeping aspect ratio (letterbox).
    """

    h0, w0 = img.shape[:2]
    r = min(new_shape[0] / h0, new_shape[1] / w0)
    new_unpad = (int(w0 * r), int(h0 * r))
    dh, dw = new_shape[0] - new_unpad[1], new_shape[1] - new_unpad[0]
    dh, dw = dh / 2, dw / 2

    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))

    if len(img.shape) == 2:
        return cv2.copyMakeBorder(
            img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color[0]
        )
    else:
        return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)


def apply_clahe(gray, clip_limit, grid_size):
    """
    Apply CLAHE on a grayscale image.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    return clahe.apply(gray)


def denoise_image(gray, h, template_size, search_size):
    """
    Denoise grayscale with NLMeans.
    """
    return cv2.fastNlMeansDenoising(gray, None, h, template_size, search_size)


def preprocess_image(img, clahe_cfg=CLAHE_CFG, denoise_cfg=DENOISE_CFG, target_size=TARGET_SIZE):
    """
    Full pipeline: detect gray→ normalize→ CLAHE→ denoise→ ensure 3ch→ letterbox.
    """
    if img.ndim == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()

    norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype("uint8")

    if clahe_cfg:
        norm = apply_clahe(norm, **clahe_cfg)

    if denoise_cfg:
        norm = denoise_image(norm, **denoise_cfg)

    bgr = cv2.cvtColor(norm, cv2.COLOR_GRAY2BGR)

    return letterbox_resize(bgr, new_shape=target_size)

In [4]:
# Batch Processing Function


def process_directory(input_dir, output_dir, exts=EXTS):
    """
    Walks through input_dir, preprocesses each image, and writes
    to output_dir mirroring folder structure.
    """
    total = 0
    for root, _, files in os.walk(input_dir):
        rel = os.path.relpath(root, input_dir)
        out_subdir = os.path.join(output_dir, rel)
        os.makedirs(out_subdir, exist_ok=True)

        for fname in files:
            if not any(fname.lower().endswith(ext) for ext in exts):
                continue
            src = os.path.join(root, fname)
            img = cv2.imread(src)
            if img is None:
                continue

            pre = preprocess_image(img)
            dst = os.path.join(out_subdir, fname)
            cv2.imwrite(dst, pre)
            total += 1

    return total

In [5]:
# Run Preprocessing


print(f"Preprocessing {INPUT_DIR} → {PREPROCESSED_OUTPUT_DIR} ...")
count = process_directory(INPUT_DIR, PREPROCESSED_OUTPUT_DIR)
print(f"Done! Processed {count} images.")

Preprocessing /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train → /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/preprocessed_data ...


KeyboardInterrupt: 

Convert raw sample data to yolo format

[Object Detection dataset overview](https://docs.ultralytics.com/datasets/detect/)

In [20]:
# Creates dataset to fit yolo format

random.seed(42)

data_root = PREPROCESSED_OUTPUT_DIR
labels_csv = config.TRAIN_LABELS_PATH

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val = os.path.join(output_base, "labels", "val")

for path in (
    output_images_train,
    output_images_val,
    output_labels_train,
    output_labels_val,
):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id = row["tomo_id"]
    z = int(row["Motor axis 0"])
    y = row["Motor axis 1"]
    x = row["Motor axis 2"]
    img_width = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name = f"slice_{z:04d}.jpg"
    img_path = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)

all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

random.shuffle(all_images)
split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]


def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z = int(os.path.splitext(img_path)[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        dest_img = os.path.join(img_out_dir, f"{base_fn}.jpg")
        dest_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")

        shutil.copyfile(img_path, dest_img)

        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            bbox_w, bbox_h = 10, 10
            cx = x / w
            cy = y / h
            nw = bbox_w / w
            nh = bbox_h / h
            with open(dest_lbl, "w") as f:
                f.write(f"0 {cx} {cy} {nw} {nh}\n")
        else:
            open(dest_lbl, "w").close()


# Process both sets
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images, output_images_val, output_labels_val)

print(f"Dataset created with {len(train_images)} training and {len(val_images)} validation images.")

KeyboardInterrupt: 

Data augmentation for training data using albumentations liblary.

In [ ]:
# Data augmentation

# 1) Defines augmentation pipeline
transform = A.Compose(
    [
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.8),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.2),
)


def load_yolo_labels(label_path):
    bboxes, class_labels = [], []
    for line in open(label_path):
        cls, x, y, w, h = map(float, line.split())
        bboxes.append([x, y, w, h])
        class_labels.append(int(cls))
    return bboxes, class_labels


# Paths for augmented data output and yolo format input
data_root = config.YOLO_DATA_DIR
output_root = config.AUGMENTED_YOLO_DATA

IMAGES_DIR = os.path.join(data_root, "images", "train")
LABELS_DIR = os.path.join(data_root, "labels", "train")

os.path.join(output_root, "images", "train")
os.path.join(output_root, "labels", "train")

OUT_IMAGES = os.path.join(output_root, "images", "train")
OUT_LABELS = os.path.join(output_root, "labels", "train")

os.makedirs(OUT_IMAGES, exist_ok=True)
os.makedirs(OUT_LABELS, exist_ok=True)

for img_name in tqdm(os.listdir(IMAGES_DIR)):
    img_path = os.path.join(IMAGES_DIR, img_name)
    label_path = os.path.join(LABELS_DIR, img_name.replace(".jpg", ".txt"))

    img = cv2.imread(img_path)
    bboxes, class_labels = load_yolo_labels(label_path)

    # apply N random augmentations per image
    for i in range(3):  # generates 3 aug versions
        transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
        aug_img = transformed["image"]
        aug_bboxes = transformed["bboxes"]
        aug_labels = transformed["class_labels"]

        out_img = os.path.join(OUT_IMAGES, f"{os.path.splitext(img_name)[0]}_aug{i}.jpg")
        cv2.imwrite(out_img, aug_img)

        out_lbl = os.path.join(OUT_LABELS, f"{os.path.splitext(img_name)[0]}_aug{i}.txt")
        with open(out_lbl, "w") as f:
            for cls, (x, y, w, h) in zip(aug_labels, aug_bboxes):
                f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

/Users/kereminci/Desktop/cms-team/BYU_Locating_Bacterial_Flagellar_Motors_2025/venv/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/3c/s9rrk0m14p5b80qglgftm0900000gn/T/ipykernel_87471/2678116614.py:7: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
/var/folders/3c/s9rrk0m14p5b80qglgftm0900000gn/T/ipykernel_87471/2678116614.py:12: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
100%|██████████| 640/640 [00:55<00:00, 11.55it/s]


In [8]:
# 1) Load your full slice‐level DataFrame
#    Assume df has one row per slice with columns ['tomo_id','slice','Number of motors','Array shape (axis 0)']
df = pd.read_csv(config.TRAIN_LABELS_PATH)

# 2) Build a slice‐level table with a binary label
df_slices = df.drop_duplicates(
    subset=["tomo_id", "Motor axis 2"]
).rename(  # one slice per motor‐axis index
    columns={"Motor axis 2": "slice", "Number of motors": "n_motors"}
)
df_slices["pos"] = df_slices["n_motors"] > 0

# 3) Split into positive and negative slices
pos_df = df_slices[df_slices["pos"]]
neg_df = df_slices[~df_slices["pos"]]

# 4) Undersample negatives to, say, 2× the positives
neg_sampled = neg_df.sample(n=min(len(neg_df), len(pos_df) * 2), random_state=42)

# 5) (Optionally) Oversample positives up to match negatives
#    e.g. double the positives
pos_oversampled = pos_df.sample(
    n=min(len(pos_df) * 2, len(neg_sampled)), replace=True, random_state=42
)

# 6) Stitch together and shuffle
balanced = pd.concat([neg_sampled, pos_oversampled]).sample(frac=1, random_state=42)

# 7) Now do your train/val split on the slice‐level IDs
train_slices, val_slices = train_test_split(
    balanced, test_size=0.2, stratify=balanced["pos"], random_state=42
)

# 8) Extract the lists you feed into your YOLO writer
train_images = train_slices.apply(lambda r: f"{r.tomo_id}_{r.slice}", axis=1).tolist()
val_images = val_slices.apply(lambda r: f"{r.tomo_id}_{r.slice}", axis=1).tolist()

print(f"Train: {len(train_images)} slices ({train_slices['pos'].mean() * 100:.1f}% positive)")
print(f" Val : {len(val_images)} slices ({val_slices['pos'].mean() * 100:.1f}% positive)")

print(train_images)
print(val_images)

Train: 457 slices (50.1% positive)
 Val : 115 slices (49.6% positive)
['tomo_0c2749_-1.0', 'tomo_08bf73_-1.0', 'tomo_4925ee_-1.0', 'tomo_2fb12d_-1.0', 'tomo_3b1cc9_-1.0', 'tomo_d0699e_-1.0', 'tomo_1cc887_192.0', 'tomo_d634b7_382.0', 'tomo_00e463_628.0', 'tomo_61e947_-1.0', 'tomo_5f1f0c_-1.0', 'tomo_f78e91_387.0', 'tomo_fbb49b_-1.0', 'tomo_bcb115_-1.0', 'tomo_7f0184_350.0', 'tomo_be4a3a_896.0', 'tomo_10c564_267.0', 'tomo_e7c195_-1.0', 'tomo_db2a10_-1.0', 'tomo_672101_581.0', 'tomo_3e6ead_-1.0', 'tomo_fc5ae4_313.0', 'tomo_fb08b5_559.0', 'tomo_c38e83_-1.0', 'tomo_381add_862.0', 'tomo_d56709_888.0', 'tomo_5f34b3_-1.0', 'tomo_648adf_-1.0', 'tomo_71ece1_416.0', 'tomo_95c0eb_-1.0', 'tomo_9d3a0e_355.0', 'tomo_226cd8_607.0', 'tomo_97876d_-1.0', 'tomo_1b82d1_677.0', 'tomo_acadd7_759.0', 'tomo_711fad_-1.0', 'tomo_1af88d_619.0', 'tomo_e2da77_-1.0', 'tomo_91beab_-1.0', 'tomo_c7b008_854.0', 'tomo_1dc5f9_-1.0', 'tomo_072a16_-1.0', 'tomo_a3ed10_-1.0', 'tomo_9f424e_-1.0', 'tomo_512f98_-1.0', 'tomo_9f91

In [10]:
# -------- CONFIGURE PATHS & HYPERPARAMETERS --------
csv_path = config.TRAIN_LABELS_PATH  # your annotation CSV
volumes_dir = config.TRAIN_DATASET_DIR  # e.g. data/volumes/<tomo_id>/slice_0001.png
out_dir = config.YOLO_DATA_DIR  # will contain train/ & val/
train_ratio = 0.8
neg_pos_ratio = 2  # #negatives per positive
box_size = 32  # square box side in px
img_ext = ".png"  # or .jpg

# -------- 1) Build slice-level table --------
df = pd.read_csv(csv_path)
slices = []
for tomo_id, g in df.groupby("tomo_id"):
    # how many slices in this volume?
    total_slices = int(g["Array shape (axis 2)"].iloc[0])
    # for each z-slice, count motors
    for z in range(total_slices):
        nm = int((g["Motor axis 2"] == z).sum())
        slices.append({"tomo_id": tomo_id, "slice": z, "num_motors": nm})
slice_df = pd.DataFrame(slices)
slice_df["is_pos"] = slice_df["num_motors"] > 0

# -------- 2) Balance pos/neg --------
pos_df = slice_df[slice_df["is_pos"]]
neg_df = slice_df[~slice_df["is_pos"]]

# undersample negatives to neg_pos_ratio * #positives
neg_keep = neg_df.sample(n=min(len(neg_df), len(pos_df) * neg_pos_ratio), random_state=42)
# optionally oversample positives to match neg_keep
pos_keep = pos_df.sample(n=len(pos_df), replace=True, random_state=42)

balanced = pd.concat([pos_keep, neg_keep]).sample(frac=1, random_state=42)

# -------- 3) Train/Val split --------
train_df, val_df = train_test_split(
    balanced, test_size=1 - train_ratio, stratify=balanced["is_pos"], random_state=42
)

# -------- 4) Prepare YOLO folders --------
for split in ["train", "val"]:
    for sub in ["images", "labels"]:
        d = os.path.join(out_dir, split, sub)
        os.makedirs(d, exist_ok=True)


# helper to write one slice
def write_slice(row, split):
    img_name = f"{row.tomo_id}_slice_{row.slice:04d}{img_ext}"
    src_img = os.path.join(volumes_dir, row.tomo_id, img_name)
    dst_img = os.path.join(out_dir, split, "images", img_name)
    # copy image
    shutil.copy(src_img, dst_img)

    # build label file
    lbl_path = os.path.join(out_dir, split, "labels", img_name.replace(img_ext, ".txt"))
    lines = []
    if row.num_motors > 0:
        motors = df[(df.tomo_id == row.tomo_id) & (df["Motor axis 2"] == row.slice)]
        # you need image width/height to normalize; here we assume square imgs
        # load one image to get size:
        from PIL import Image

        w, h = Image.open(src_img).size
        for _, m in motors.iterrows():
            x_c = m["Motor axis 0"]
            y_c = m["Motor axis 1"]
            # center-normalized
            x = x_c / w
            y = y_c / h
            # box normalized
            bw = box_size / w
            bh = box_size / h
            lines.append(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, "w") as f:
        f.write("\n".join(lines))


# write out
for _, row in train_df.iterrows():
    write_slice(row, "train")
for _, row in val_df.iterrows():
    write_slice(row, "val")

print(f"► Done! Train: {len(train_df)} slices, Val: {len(val_df)} slices")

FileNotFoundError: [Errno 2] No such file or directory: '/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_f78e91/tomo_f78e91_slice_0387.png'

In [16]:
# Updated slice-writing code with file-existence checks


# Configuration (adjust as needed)
csv_path = config.TRAIN_LABELS_PATH
volumes_dir = config.TRAIN_DATASET_DIR
out_dir = config.YOLO_DATA_DIR
img_ext = ".jpg"
box_size = 32

# Load annotation CSV
df = pd.read_csv(csv_path)

# Build slice-level DataFrame
slices = []
for tomo_id, g in df.groupby("tomo_id"):
    total_slices = int(g["Array shape (axis 2)"].iloc[0])
    for z in range(total_slices):
        nm = int((g["Motor axis 2"] == z).sum())
        slices.append({"tomo_id": tomo_id, "slice": z, "num_motors": nm})
slice_df = pd.DataFrame(slices)

# Balance & split (example 80/20, 2:1 neg:pos)
pos_df = slice_df[slice_df["num_motors"] > 0]
neg_df = slice_df[slice_df["num_motors"] == 0]
neg_keep = neg_df.sample(n=min(len(neg_df), len(pos_df) * 2), random_state=42)
pos_keep = pos_df.sample(n=len(pos_df), replace=True, random_state=42)
balanced = pd.concat([pos_keep, neg_keep]).sample(frac=1, random_state=42)

train_df, val_df = train_test_split(
    balanced, test_size=0.2, stratify=balanced["num_motors"] > 0, random_state=42
)

# Prepare YOLO folders
for split in ["train", "val"]:
    for sub in ["images", "labels"]:
        os.makedirs(os.path.join(out_dir, split, sub), exist_ok=True)


# Function to write one slice, skipping missing files
def write_slice(row, split):
    img_name = f"{row.tomo_id}_slice_{row.slice:04d}{img_ext}"
    src_img = os.path.join(volumes_dir, row.tomo_id, img_name)
    dst_img = os.path.join(out_dir, split, "images", img_name)
    lbl_path = os.path.join(out_dir, split, "labels", img_name.replace(img_ext, ".txt"))

    print(src_img + "  " + dst_img)

    # Skip if source image doesn't exist
    if not os.path.isfile(src_img):
        print(f"⚠️ Missing image, skipping: {src_img}")
        return

    # Copy image
    shutil.copy(src_img, dst_img)

    # Write label file
    lines = []
    if row.num_motors > 0:
        motors = df[(df.tomo_id == row.tomo_id) & (df["Motor axis 2"] == row.slice)]
        w, h = Image.open(src_img).size
        for _, m in motors.iterrows():
            x_c, y_c = m["Motor axis 0"], m["Motor axis 1"]
            x = x_c / w
            y = y_c / h
            bw = box_size / w
            bh = box_size / h
            lines.append(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, "w") as f:
        f.write("\n".join(lines))


# Write out train and val slices
for _, row in train_df.iterrows():
    write_slice(row, "train")
for _, row in val_df.iterrows():
    write_slice(row, "val")

print("✅ Preprocessing complete. Slices written with missing files skipped.")

/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_f78e91/tomo_f78e91_slice_0387.jpg  /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/yolo/train/images/tomo_f78e91_slice_0387.jpg
⚠️ Missing image, skipping: /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_f78e91/tomo_f78e91_slice_0387.jpg
/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_d0c025/tomo_d0c025_slice_0339.jpg  /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/yolo/train/images/tomo_d0c025_slice_0339.jpg
⚠️ Missing image, skipping: /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_d0c025/tomo_d0c025_slice_0339.jpg
/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_bc143f/tomo_bc143f_slice_0142.jpg  /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/yolo/train/images/tomo_bc143f_slice_0142.jpg
⚠️ Missing image, skipping: /home/h4/

In [18]:
# Configuration (adjust as needed)
csv_path = config.TRAIN_LABELS_PATH
volumes_dir = config.TRAIN_DATASET_DIR
out_dir = config.YOLO_DATA_DIR
img_ext = ".jpg"
box_size = 32

# Load annotation CSV
df = pd.read_csv(csv_path)

# Build slice-level DataFrame
slices = []
for tomo_id, g in df.groupby("tomo_id"):
    total_slices = int(g["Array shape (axis 2)"].iloc[0])
    for z in range(total_slices):
        nm = int((g["Motor axis 2"] == z).sum())
        slices.append({"tomo_id": tomo_id, "slice": z, "num_motors": nm})
slice_df = pd.DataFrame(slices)

# Example balancing and train/val split (adjust to your pipeline)
# ... (assume train_df and val_df are defined here)

# Prepare YOLO folders
for split in ["train", "val"]:
    for sub in ["images", "labels"]:
        os.makedirs(os.path.join(out_dir, split, sub), exist_ok=True)


# Function to write one slice, handling filename layout
def write_slice(row, split):
    # source filename is just slice_{:04d}.jpg inside tomo folder
    src_img_name = f"slice_{row.slice:04d}{img_ext}"
    src_img_path = os.path.join(volumes_dir, row.tomo_id, src_img_name)

    # destination uses combined naming for uniqueness
    dst_img_name = f"{row.tomo_id}_slice_{row.slice:04d}{img_ext}"
    dst_img_path = os.path.join(out_dir, split, "images", dst_img_name)

    # label file path
    lbl_path = os.path.join(out_dir, split, "labels", dst_img_name.replace(img_ext, ".txt"))

    print(src_img_name + "  " + src_img_path)

    # Skip missing files
    if not os.path.isfile(src_img_path):
        print(f"⚠️ Missing image, skipping: {src_img_path}")
        return

    # Copy image
    shutil.copy(src_img_path, dst_img_path)

    # Write label file
    lines = []
    if row.num_motors > 0:
        motors = df[(df.tomo_id == row.tomo_id) & (df["Motor axis 2"] == row.slice)]
        w, h = Image.open(src_img_path).size
        for _, m in motors.iterrows():
            x_c, y_c = m["Motor axis 0"], m["Motor axis 1"]
            x = x_c / w
            y = y_c / h
            bw = box_size / w
            bh = box_size / h
            lines.append(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, "w") as f:
        f.write("\n".join(lines))


# Example usage:
# for _, row in train_df.iterrows():
#     write_slice(row, 'train')
# for _, row in val_df.iterrows():
#     write_slice(row, 'val')

In [1]:
# Creates dataset to fit yolo format

random.seed(42)

data_root = config.TRAIN_DATASET_DIR
labels_csv = config.FILTERED_TRAIN_LABELS

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val = os.path.join(output_base, "labels", "val")

box_size = 32


for path in (
    output_images_train,
    output_images_val,
    output_labels_train,
    output_labels_val,
):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id = row["tomo_id"]
    z = int(row["Motor axis 0"])
    y = row["Motor axis 1"]
    x = row["Motor axis 2"]
    img_width = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name = f"slice_{z:04d}.jpg"
    img_path = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)

all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

# 1) Configuration for balancing
neg_pos_ratio = 1  # number of negative samples per positive

# 2) Separate positive & negative image paths
pos_images = list(positive_samples.keys())
neg_images = [img for img in all_images if img not in positive_samples]

# 3) Undersample negatives to at most neg_pos_ratio * #positives
num_pos = len(pos_images)
num_neg_keep = min(len(neg_images), num_pos * neg_pos_ratio)
neg_keep = random.sample(neg_images, num_neg_keep)

# 4) Combine & shuffle
balanced_images = pos_images + neg_keep
random.shuffle(balanced_images)

# 5) Train/val split on the balanced set
split_idx = int(len(balanced_images) * 0.8)
train_images = balanced_images[:split_idx]
val_images = balanced_images[split_idx:]


def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        # 1) Skip if the source image file doesn't exist
        if not os.path.isfile(img_path):
            print(f"⚠️  Skipping missing image: {img_path}")
            continue

        # 2) Reconstruct tomo_id and slice index as before
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z = int(os.path.splitext(os.path.basename(img_path))[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"

        dst_img = os.path.join(img_out_dir, f"{base_fn}.jpg")
        dst_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")

        # 3) Copy the image
        shutil.copy(img_path, dst_img)

        # 4) Write the label file (empty for negatives)
        lines = []
        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            cx = x / w
            cy = y / h
            bw = box_size * 2 / w
            bh = box_size * 2 / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        with open(dst_lbl, "w") as f:
            f.write("\n".join(lines))


# Process both sets
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images, output_images_val, output_labels_val)

print(f"Dataset created with {len(train_images)} training and {len(val_images)} validation images.")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


NameError: name 'random' is not defined

In [5]:
# 1) Set paths – adjust these!
root_dir = config.TRAIN_DATASET_DIR  # parent folder containing tomo_* subfolders
labels_csv = config.TRAIN_LABELS_PATH  # original CSV with a 'tomo_id' column
output_csv = config.FILTERED_TRAIN_LABELS  # where to write the filtered CSV

print(os.listdir(root_dir))
# 2) Detect all tomo_ids by listing subfolders
tomo_ids = [name for name in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, name))]
print(f"Detected {len(tomo_ids)} tomo_id folders.")
print(tomo_ids)

# 3) Load your full labels CSV
df = pd.read_csv(labels_csv)

# 4) Filter rows where 'tomo_id' is in the detected folders
filtered_df = df[df["tomo_id"].isin(tomo_ids)]
print(f"Filtering yields {len(filtered_df)} rows out of {len(df)} total.")

# 5) Write out the new CSV
filtered_df.to_csv(output_csv, index=False)
print(f"Written filtered CSV to {output_csv}")

# 6) (Optional) Preview first few rows
print(filtered_df.head())

['tomo_3183d2', 'tomo_1af88d', 'tomo_0363f2', 'tomo_1446aa', 'tomo_8d231b', 'tomo_04d42b', 'tomo_6df2d6', 'tomo_56b9a3', 'tomo_3e7783', 'tomo_2483bb', 'tomo_0a8f05', 'tomo_49725c', 'tomo_1e9980', 'tomo_935ae0', 'tomo_79756f', 'tomo_30b580', 'tomo_072a16', 'tomo_6bc974', 'tomo_79a385', 'tomo_4469a7', 'tomo_9c0253', 'tomo_3c6038', 'tomo_88af60', 'tomo_62eea8', '.DS_Store', 'tomo_2e1f4c']
Detected 25 tomo_id folders.
['tomo_3183d2', 'tomo_1af88d', 'tomo_0363f2', 'tomo_1446aa', 'tomo_8d231b', 'tomo_04d42b', 'tomo_6df2d6', 'tomo_56b9a3', 'tomo_3e7783', 'tomo_2483bb', 'tomo_0a8f05', 'tomo_49725c', 'tomo_1e9980', 'tomo_935ae0', 'tomo_79756f', 'tomo_30b580', 'tomo_072a16', 'tomo_6bc974', 'tomo_79a385', 'tomo_4469a7', 'tomo_9c0253', 'tomo_3c6038', 'tomo_88af60', 'tomo_62eea8', 'tomo_2e1f4c']
Filtering yields 26 rows out of 737 total.
Written filtered CSV to /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/filtered_train_labels.csv
    row_id      tomo_id  Motor axis 0  Mo